# Verificando o path dos csvs

In [0]:
display(dbutils.fs.ls("dbfs:/Volumes/atividade2/atividade2_catalogo/atividade2-volume"))

# Importando as bibliotecas

In [0]:
from pyspark.sql.functions import current_timestamp
import requests
from pyspark.sql.functions import current_timestamp
from datetime import datetime

# Criando as duas databases

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS bronze;
CREATE DATABASE IF NOT EXISTS silver;

# Lendo cada csv

In [0]:
# Base path inside your Unity Catalog volume
base_path = "dbfs:/Volumes/atividade2/atividade2_catalogo/atividade2-volume/"

# Mapping of file names to Bronze table names
files_to_tables = {
    "olist_customers_dataset.csv": "bronze.ft_consumidores",
    "olist_geolocation_dataset.csv": "bronze.ft_geolocalizacao",
    "olist_order_items_dataset.csv": "bronze.ft_itens_pedidos",
    "olist_order_payments_dataset.csv": "bronze.ft_pagamentos_pedidos",
    "olist_order_reviews_dataset.csv": "bronze.ft_avaliacoes_pedidos",
    "olist_orders_dataset.csv": "bronze.ft_pedidos",
    "olist_products_dataset.csv": "bronze.ft_produtos",
    "olist_sellers_dataset.csv": "bronze.ft_vendedores",
    "product_category_name_translation.csv": "bronze.dm_categoria_produtos_traducao"
}

In [0]:
for file_name, table_name in files_to_tables.items():
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{base_path}{file_name}")
        .withColumn("ingestion_timestamp", current_timestamp())
    )
    df.write.mode("overwrite").saveAsTable(table_name)
    print(f" Table created: {table_name}")

# Adicionando a informação sobre o dolar

In [0]:
data_inicio_formatada = "01-01-2016"
data_fim_formatada = "01-01-2020"

# API URL
url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(\
dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'\
&$select=dataHoraCotacao,cotacaoCompra&$format=json"

# Request
response = requests.get(url).json()

df_dollar = spark.createDataFrame(response["value"])
df_dollar = df_dollar.withColumn("ingestion_timestamp", current_timestamp())

# Salvando a bronze
df_dollar.write.mode("overwrite").saveAsTable("bronze.dm_cotacao_dolar")

display(df_dollar.limit(10))

In [0]:
%sql
SHOW TABLES IN bronze;